In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import cm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.cm import get_cmap
from matplotlib.animation import FuncAnimation, FFMpegWriter
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from mvlearn.embed import GCCA
from channel import ChannelReader
import xlrd
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import make_interp_spline
from scipy.signal import correlate
from scipy.linalg import orthogonal_procrustes
from scipy.linalg import inv
from numpy.linalg import svd, qr
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr
from itertools import combinations

In [ ]:
output_dir = "D:/상훈/Decoding/250804_은지/cca-pca 6개"  # 출력 디렉토리
file_basename = ""
bin_size_ms = 80

In [ ]:
stimulus1 = "A1"
stimulus2 = "A2" 
stimulus3 = "A3" 
stimulus4 = "A4" 
stimulus5 = "A5" 
stimulus6 = "A6" 
stimulus7 = "B1"
stimulus8 = "B2" 
stimulus9 = "B3" 
stimulus10 = "B4" 
stimulus11 = "B5" 
stimulus12 = "B6" 

data1 = {stimulus1: } 
data2 = {stimulus2: }
data3 = {stimulus3: } 
data4 = {stimulus4: }
data5 = {stimulus5: }
data6 = {stimulus6: }
data7 = {stimulus7: }
data8 = {stimulus8: }
data9 = {stimulus9: }
data10 = {stimulus10: }
data11 = {stimulus11: }
data12 = {stimulus12: }



data_dict = {**data1, **data2, **data3, **data4, **data5, **data6, **data7, **data8, **data9, **data10, **data11, **data12}

data_dict = {stimulus1: [data1[stimulus1]], stimulus2: [data2[stimulus2]], stimulus3: [data3[stimulus3]], stimulus4: [data4[stimulus4]], stimulus5: [data5[stimulus5]], stimulus6: [data6[stimulus6]], stimulus7: [data7[stimulus7]], stimulus8: [data8[stimulus8]], stimulus9: [data9[stimulus9]], stimulus10: [data10[stimulus10]], stimulus11: [data11[stimulus11]], stimulus12: [data12[stimulus12]]}

In [ ]:
def calculate_trajectory_length(trajectory):
    distances = np.sqrt(np.sum(np.diff(trajectory, axis=0)**2, axis=1))
    total_length = np.sum(distances)
    return total_length

# Curvature 계산
def calculate_curvature(trajectory):
    curvatures = []
    for i in range(1, len(trajectory) - 1):
        p1, p2, p3 = trajectory[i-1], trajectory[i], trajectory[i+1]
        v1 = p2 - p1
        v2 = p3 - p2
        norm_v1, norm_v2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if norm_v1 == 0 or norm_v2 == 0:
            continue
        cos_theta = np.dot(v1, v2) / (norm_v1 * norm_v2)
        theta = np.arccos(np.clip(cos_theta, -1, 1))
        curvature = theta / (norm_v1 + norm_v2)
        curvatures.append(curvature)
    return np.mean(curvatures) if curvatures else 0

# Rate of Change 계산
def calculate_rate_of_change(trajectory, bin_size_ms):
    distances = np.sqrt(np.sum(np.diff(trajectory, axis=0)**2, axis=1))
    time_step = bin_size_ms / 1000
    rates = distances / time_step
    return np.mean(rates), np.std(rates)

# Participation Ratio (PR)
def calculate_participation_ratio(pca):
    eigenvalues = pca.explained_variance_
    return (np.sum(eigenvalues) ** 2) / np.sum(eigenvalues ** 2)

# Intrinsic Dimensionality (MLE-based)
def estimate_intrinsic_dimension_mle(data, k=5):
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(data)
    distances, _ = nbrs.kneighbors(data)
    distances = np.maximum(distances[:, 1:], 1e-10)
    log_ratios = np.log(distances[:, -1:] / distances[:, :-1])
    S_k = np.sum(log_ratios, axis=1)
    mle = (k - 2) / S_k
    return np.mean(mle[mle > 0])

# Lyapunov Exponent
def calculate_lyapunov_exponent(data, embedding_dim=3, time_delay=1, min_dist=1e-10):
    n_vectors = len(data) - (embedding_dim - 1) * time_delay
    if n_vectors <= 0:
        return np.nan

    reconstructed = np.zeros((n_vectors, embedding_dim * data.shape[1]))
    for i in range(n_vectors):
        for j in range(embedding_dim):
            reconstructed[i, j*data.shape[1]:(j+1)*data.shape[1]] = data[i + j*time_delay]

    nbrs = NearestNeighbors(n_neighbors=2).fit(reconstructed)
    distances, indices = nbrs.kneighbors(reconstructed)

    divergence = []
    max_steps = min(5, n_vectors - 1)
    for t in range(max_steps):
        dist_t = [np.log(max(np.linalg.norm(reconstructed[i+t] - reconstructed[indices[i,1]+t]), min_dist))
                  for i in range(n_vectors - t) if indices[i,1] < n_vectors - t]
        if dist_t:
            divergence.append(np.mean(dist_t))

    if len(divergence) < 2:
        return np.nan

    lyap_exp = np.polyfit(np.arange(len(divergence)), divergence, 1)[0]
    return lyap_exp

In [ ]:
firing_matrix1 = np.array(data_dict[stimulus1])  
firing_matrix2 = np.array(data_dict[stimulus2])  
firing_matrix3 = np.array(data_dict[stimulus3])
firing_matrix4 = np.array(data_dict[stimulus4])  
firing_matrix5 = np.array(data_dict[stimulus5])  
firing_matrix6 = np.array(data_dict[stimulus6])
firing_matrix7 = np.array(data_dict[stimulus7])  
firing_matrix8 = np.array(data_dict[stimulus8])  
firing_matrix9 = np.array(data_dict[stimulus9])
firing_matrix10 = np.array(data_dict[stimulus10])
firing_matrix11 = np.array(data_dict[stimulus11])
firing_matrix12 = np.array(data_dict[stimulus12])

mean_firing1 = np.mean(firing_matrix1, axis=0)  
mean_firing2 = np.mean(firing_matrix2, axis=0)
mean_firing3 = np.mean(firing_matrix3, axis=0)
mean_firing4 = np.mean(firing_matrix4, axis=0)  
mean_firing5 = np.mean(firing_matrix5, axis=0)
mean_firing6 = np.mean(firing_matrix6, axis=0)
mean_firing7 = np.mean(firing_matrix7, axis=0)  
mean_firing8 = np.mean(firing_matrix8, axis=0)
mean_firing9 = np.mean(firing_matrix9, axis=0)
mean_firing10 = np.mean(firing_matrix10, axis=0)  
mean_firing11 = np.mean(firing_matrix11, axis=0)
mean_firing12 = np.mean(firing_matrix12, axis=0)


# Cosine similarity function
def cosine_similarity(v1, v2):
    return 1 - cosine(v1, v2)

# Apply Gaussian smoothing
smoothed_firing1 = gaussian_filter1d(mean_firing1, sigma=3, axis=1)
smoothed_firing2 = gaussian_filter1d(mean_firing2, sigma=3, axis=1)
smoothed_firing3 = gaussian_filter1d(mean_firing3, sigma=3, axis=1)
smoothed_firing4 = gaussian_filter1d(mean_firing4, sigma=3, axis=1)
smoothed_firing5 = gaussian_filter1d(mean_firing5, sigma=3, axis=1)
smoothed_firing6 = gaussian_filter1d(mean_firing6, sigma=3, axis=1)
smoothed_firing7 = gaussian_filter1d(mean_firing7, sigma=3, axis=1)
smoothed_firing8 = gaussian_filter1d(mean_firing8, sigma=3, axis=1)
smoothed_firing9 = gaussian_filter1d(mean_firing9, sigma=3, axis=1)
smoothed_firing10 = gaussian_filter1d(mean_firing10, sigma=3, axis=1)
smoothed_firing11 = gaussian_filter1d(mean_firing11, sigma=3, axis=1)
smoothed_firing12 = gaussian_filter1d(mean_firing12, sigma=3, axis=1)

X1 = smoothed_firing1.T  
X2 = smoothed_firing2.T  
X3 = smoothed_firing3.T 
X4 = smoothed_firing4.T  
X5 = smoothed_firing5.T  
X6 = smoothed_firing6.T 
X7 = smoothed_firing7.T  
X8 = smoothed_firing8.T  
X9 = smoothed_firing9.T 
X10 = smoothed_firing10.T  
X11 = smoothed_firing11.T  
X12 = smoothed_firing12.T 

# Print shapes
print(f"Shape of matrix of {stimulus1}: {firing_matrix1.shape}")
print(f"Shape of matrix of {stimulus2}: {firing_matrix2.shape}")
print(f"Shape of matrix of {stimulus3}: {firing_matrix3.shape}")
print(f"Shape of matrix of {stimulus4}: {firing_matrix4.shape}")
print(f"Shape of matrix of {stimulus5}: {firing_matrix5.shape}")
print(f"Shape of matrix of {stimulus6}: {firing_matrix6.shape}")
print(f"Shape of matrix of {stimulus7}: {firing_matrix7.shape}")
print(f"Shape of matrix of {stimulus8}: {firing_matrix8.shape}")
print(f"Shape of matrix of {stimulus9}: {firing_matrix9.shape}")
print(f"Shape of matrix of {stimulus10}: {firing_matrix10.shape}")
print(f"Shape of matrix of {stimulus11}: {firing_matrix11.shape}")
print(f"Shape of matrix of {stimulus12}: {firing_matrix12.shape}")
print()



In [ ]:
min_len = min(X1.shape[0], X2.shape[0], X3.shape[0], X4.shape[0], X5.shape[0], X6.shape[0], X7.shape[0], X8.shape[0], X9.shape[0], X10.shape[0], X11.shape[0], X12.shape[0])

X1 = X1[:min_len]
X2 = X2[:min_len]
X3 = X3[:min_len]
X4 = X4[:min_len]
X5 = X5[:min_len]
X6 = X6[:min_len]
X7 = X7[:min_len]
X8 = X8[:min_len]
X9 = X9[:min_len]
X10 = X10[:min_len]
X11 = X11[:min_len]
X12 = X12[:min_len]

n_components_pca = 10
pca = PCA(n_components=n_components_pca)

X1_pca = pca.fit_transform(X1)
X2_pca = pca.fit_transform(X2)
X3_pca = pca.fit_transform(X3)
X4_pca = pca.fit_transform(X4)
X5_pca = pca.fit_transform(X5)
X6_pca = pca.fit_transform(X6)
X7_pca = pca.fit_transform(X7)
X8_pca = pca.fit_transform(X8)
X9_pca = pca.fit_transform(X9)
X10_pca = pca.fit_transform(X10)
X11_pca = pca.fit_transform(X11)
X12_pca = pca.fit_transform(X12)

# GCCA to common latent space
gcca = GCCA(n_components=3)
X1_aligned, X2_aligned, X3_aligned, X4_aligned, X5_aligned, X6_aligned = gcca.fit_transform([X1_pca, X2_pca, X3_pca, X4_pca, X5_pca, X6_pca])
Y1_aligned, Y2_aligned, Y3_aligned, Y4_aligned, Y5_aligned, Y6_aligned = gcca.fit_transform([X7_pca, X8_pca, X9_pca, X10_pca, X11_pca, X12_pca])
#Z1_aligned, Z2_aligned, Z3_aligned = gcca.fit_transform([X7_pca, X8_pca, X9_pca])

# X1_aligned, X2_aligned, X3_aligned are all shape (n_time_bins, 3)
# and aligned in the same canonical latent space

#########################
# Visualization_stim3

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 1~6 트라이얼(같은 계열색: Blues)
trial_colors_1to6 = ['#9ecae1', '#6baed6', '#4292c6', '#2171b5', '#08519c', '#08306b']

ax.plot(X1_aligned[:, 0], X1_aligned[:, 1], X1_aligned[:, 2], linewidth=2, color='#9ecae1', label=f'{stimulus1}')
for i in range(1, len(X1_aligned)):
    ax.plot([X1_aligned[i-1, 0], X1_aligned[i, 0]], 
            [X1_aligned[i-1, 1], X1_aligned[i, 1]], 
            [X1_aligned[i-1, 2], X1_aligned[i, 2]], 
            color=plt.cm.Blues(i / len(X1_aligned)))

ax.plot(X2_aligned[:, 0], X2_aligned[:, 1], X2_aligned[:, 2], linewidth=2, color='#6baed6', label=f'{stimulus2}')
for i in range(1, len(X2_aligned)):
    ax.plot([X2_aligned[i-1, 0], X2_aligned[i, 0]], 
            [X2_aligned[i-1, 1], X2_aligned[i, 1]], 
            [X2_aligned[i-1, 2], X2_aligned[i, 2]], 
            color=plt.cm.Reds(i / len(X2_aligned)))
    
ax.plot(X3_aligned[:, 0], X3_aligned[:, 1], X3_aligned[:, 2], linewidth=2, color='#4292c6', label=f'{stimulus3}')
for i in range(1, len(X3_aligned)):
    ax.plot([X3_aligned[i-1, 0], X3_aligned[i, 0]], 
            [X3_aligned[i-1, 1], X3_aligned[i, 1]], 
            [X3_aligned[i-1, 2], X3_aligned[i, 2]], 
            color=plt.cm.Greens(i / len(X3_aligned)))

ax.plot(X4_aligned[:, 0], X4_aligned[:, 1], X4_aligned[:, 2], linewidth=2, color='#2171b5', label=f'{stimulus4}')
for i in range(1, len(X4_aligned)):
    ax.plot([X4_aligned[i-1, 0], X4_aligned[i, 0]],
            [X4_aligned[i-1, 1], X4_aligned[i, 1]],
            [X4_aligned[i-1, 2], X4_aligned[i, 2]],
            color=plt.cm.Oranges(i / len(X4_aligned)))

ax.plot(X5_aligned[:, 0], X5_aligned[:, 1], X5_aligned[:, 2], linewidth=2, color='#08519c', label=f'{stimulus5}')
for i in range(1, len(X5_aligned)):
    ax.plot([X5_aligned[i-1, 0], X5_aligned[i, 0]],
            [X5_aligned[i-1, 1], X5_aligned[i, 1]],
            [X5_aligned[i-1, 2], X5_aligned[i, 2]],
            color=plt.cm.Purples(i / len(X5_aligned)))

ax.plot(X6_aligned[:, 0], X6_aligned[:, 1], X6_aligned[:, 2], linewidth=2, color='#08306b', label=f'{stimulus6}')
for i in range(1, len(X6_aligned)):
    ax.plot([X6_aligned[i-1, 0], X6_aligned[i, 0]],
            [X6_aligned[i-1, 1], X6_aligned[i, 1]],
            [X6_aligned[i-1, 2], X6_aligned[i, 2]],
            color=plt.cm.BuGn(i / len(X6_aligned)))            

ax.set_title(f"Multiset CCA-aligned Latent Dynamics_stim3")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()

# Save figure
filename = f"stim3_multi CCA alignment_3D_LD.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()

#########################
# Visualization_stim4

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 7~12 트라이얼(같은 계열색: Oranges)  # 원하면 Greens로 바꿔도 됨
trial_colors_7to12 = ['#fdd0a2', '#fdae6b', '#fd8d3c', '#f16913', '#d94801', '#a63603']

ax.plot(Y1_aligned[:, 0], Y1_aligned[:, 1], Y1_aligned[:, 2], linewidth=2, color='#fdd0a2', label=f'{stimulus7}')
for i in range(1, len(Y1_aligned)):
    ax.plot([Y1_aligned[i-1, 0], Y1_aligned[i, 0]], 
            [Y1_aligned[i-1, 1], Y1_aligned[i, 1]], 
            [Y1_aligned[i-1, 2], Y1_aligned[i, 2]], 
            color=plt.cm.Blues(i / len(Y1_aligned)))

ax.plot(Y2_aligned[:, 0], Y2_aligned[:, 1], Y2_aligned[:, 2], linewidth=2, color='#fdae6b', label=f'{stimulus8}')
for i in range(1, len(Y2_aligned)):
    ax.plot([Y2_aligned[i-1, 0], Y2_aligned[i, 0]], 
            [Y2_aligned[i-1, 1], Y2_aligned[i, 1]], 
            [Y2_aligned[i-1, 2], Y2_aligned[i, 2]], 
            color=plt.cm.Reds(i / len(Y2_aligned)))
    
ax.plot(Y3_aligned[:, 0], Y3_aligned[:, 1], Y3_aligned[:, 2], linewidth=2, color='#fd8d3c', label=f'{stimulus9}')
for i in range(1, len(Y3_aligned)):
    ax.plot([Y3_aligned[i-1, 0], Y3_aligned[i, 0]], 
            [Y3_aligned[i-1, 1], Y3_aligned[i, 1]], 
            [Y3_aligned[i-1, 2], Y3_aligned[i, 2]], 
            color=plt.cm.Greens(i / len(Y3_aligned)))

ax.plot(Y4_aligned[:, 0], Y4_aligned[:, 1], Y4_aligned[:, 2], linewidth=2, color='#f16913', label=f'{stimulus10}')
for i in range(1, len(Y4_aligned)):
    ax.plot([Y4_aligned[i-1, 0], Y4_aligned[i, 0]], 
            [Y4_aligned[i-1, 1], Y4_aligned[i, 1]], 
            [Y4_aligned[i-1, 2], Y4_aligned[i, 2]], 
            color=plt.cm.Blues(i / len(Y4_aligned)))

ax.plot(Y5_aligned[:, 0], Y5_aligned[:, 1], Y5_aligned[:, 2], linewidth=2, color='#d94801', label=f'{stimulus11}')
for i in range(1, len(Y5_aligned)):
    ax.plot([Y5_aligned[i-1, 0], Y5_aligned[i, 0]], 
            [Y5_aligned[i-1, 1], Y5_aligned[i, 1]], 
            [Y5_aligned[i-1, 2], Y5_aligned[i, 2]], 
            color=plt.cm.Reds(i / len(Y5_aligned)))
    
ax.plot(Y6_aligned[:, 0], Y6_aligned[:, 1], Y6_aligned[:, 2], linewidth=2, color='#a63603', label=f'{stimulus12}')
for i in range(1, len(Y6_aligned)):
    ax.plot([Y6_aligned[i-1, 0], Y6_aligned[i, 0]], 
            [Y6_aligned[i-1, 1], Y6_aligned[i, 1]], 
            [Y6_aligned[i-1, 2], Y6_aligned[i, 2]], 
            color=plt.cm.Greens(i / len(Y6_aligned)))    
    
    

ax.set_title(f"Multiset CCA-aligned Latent Dynamics_stim4")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()

# Save figure
filename = f"stim4_multi CCA alignment_3D_LD.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()



In [ ]:
data_dict = {stimulus1: X1_aligned, stimulus2: X2_aligned, stimulus3: X3_aligned, stimulus4: X4_aligned, stimulus5: X5_aligned, stimulus6: X6_aligned}

results=[]

for stimulus, X_pca in data_dict.items():
    print(f"\n=== CCA alignment {stimulus.upper()} ===")

    traj_length = calculate_trajectory_length(X_pca)
    print(f"Trajectory Length: {traj_length:.4f}")

    avg_curvature = calculate_curvature(X_pca)
    print(f"Average Curvature: {avg_curvature:.4f}")

    mean_rate, std_rate = calculate_rate_of_change(X_pca, bin_size_ms)
    print(f"Mean Rate of Change: {mean_rate:.4f}, Std: {std_rate:.4f}")

    pca_model = PCA().fit(X_pca)
    pr = calculate_participation_ratio(pca_model)
    print(f"Participation Ratio (PR): {pr:.4f}")

    id_mle = estimate_intrinsic_dimension_mle(X_pca)
    print(f"Intrinsic Dimensionality (MLE): {id_mle:.4f}")

    autocorr = correlate(X_pca[:, 0], X_pca[:, 0], mode='full')[len(X_pca)-1:]
    tau = np.argmax(autocorr < autocorr[0] / np.e) or 1
    print(f"Estimated time delay (tau): {tau}")

    lyap_exp = calculate_lyapunov_exponent(X_pca, embedding_dim=3, time_delay=tau)
    print(f"Lyapunov Exponent: {lyap_exp:.4f}")

    # Lyapunov 지수 해석 추가
    if np.isnan(lyap_exp):
        print("Lyapunov Exponent 계산 불가: 데이터가 너무 짧거나 파라미터 조정 필요.")
    elif lyap_exp > 0:
        print("양수: 시스템이 혼돈적이며 예측 불가능한 경향이 있음.")
    elif lyap_exp == 0:
        print("0: 시스템이 중립적이며 안정적인 상태 (주기적 궤적 가능).")
    else:
        print("음수: 시스템이 안정적이며 예측 가능함.")

    results.append({
        'Stimulus': stimulus,
        'Trajectory Length': traj_length,
        'Curvature': avg_curvature,
        'Rate of Change (mean)': mean_rate,
        'Rate of Change (std)': std_rate,
        'Participation Ratio': pr,
        'Intrinsic Dim (MLE)': id_mle,
        'Tau (delay)': tau,
        'Lyapunov Exponent': lyap_exp
    })

df_results = pd.DataFrame(results)

# p.xlsx 저장
excel_path = os.path.join(output_dir, f'stim3_multi-CCA alignment_features.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    #cca_summary.to_excel(writer, index=False, sheet_name='CCA properties')
    df_results.to_excel(writer, index=False, sheet_name='Features')
    
print()
print(f"모든 feature 결과를 {excel_path} 에 저장했습니다.")    

# Feature extraction
data_dict = {stimulus7: Y1_aligned, stimulus8: Y2_aligned, stimulus9: Y3_aligned, stimulus10: Y4_aligned, stimulus11: Y5_aligned, stimulus12: Y6_aligned}

results=[]

for stimulus, X_pca in data_dict.items():
    print(f"\n=== CCA alignment {stimulus.upper()} ===")

    traj_length = calculate_trajectory_length(X_pca)
    print(f"Trajectory Length: {traj_length:.4f}")

    avg_curvature = calculate_curvature(X_pca)
    print(f"Average Curvature: {avg_curvature:.4f}")

    mean_rate, std_rate = calculate_rate_of_change(X_pca, bin_size_ms)
    print(f"Mean Rate of Change: {mean_rate:.4f}, Std: {std_rate:.4f}")

    pca_model = PCA().fit(X_pca)
    pr = calculate_participation_ratio(pca_model)
    print(f"Participation Ratio (PR): {pr:.4f}")

    id_mle = estimate_intrinsic_dimension_mle(X_pca)
    print(f"Intrinsic Dimensionality (MLE): {id_mle:.4f}")

    autocorr = correlate(X_pca[:, 0], X_pca[:, 0], mode='full')[len(X_pca)-1:]
    tau = np.argmax(autocorr < autocorr[0] / np.e) or 1
    print(f"Estimated time delay (tau): {tau}")

    lyap_exp = calculate_lyapunov_exponent(X_pca, embedding_dim=3, time_delay=tau)
    print(f"Lyapunov Exponent: {lyap_exp:.4f}")

    # Lyapunov 지수 해석 추가
    if np.isnan(lyap_exp):
        print("Lyapunov Exponent 계산 불가: 데이터가 너무 짧거나 파라미터 조정 필요.")
    elif lyap_exp > 0:
        print("양수: 시스템이 혼돈적이며 예측 불가능한 경향이 있음.")
    elif lyap_exp == 0:
        print("0: 시스템이 중립적이며 안정적인 상태 (주기적 궤적 가능).")
    else:
        print("음수: 시스템이 안정적이며 예측 가능함.")

    results.append({
        'Stimulus': stimulus,
        'Trajectory Length': traj_length,
        'Curvature': avg_curvature,
        'Rate of Change (mean)': mean_rate,
        'Rate of Change (std)': std_rate,
        'Participation Ratio': pr,
        'Intrinsic Dim (MLE)': id_mle,
        'Tau (delay)': tau,
        'Lyapunov Exponent': lyap_exp
    })

df_results = pd.DataFrame(results)

# p.xlsx 저장
excel_path = os.path.join(output_dir, f'stim4_multi-CCA alignment_features.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    #cca_summary.to_excel(writer, index=False, sheet_name='CCA properties')
    df_results.to_excel(writer, index=False, sheet_name='Features')
    
print()
print(f"모든 feature 결과를 {excel_path} 에 저장했습니다.") 


In [ ]:
X_aligned_trials = [X1_aligned, X2_aligned, X3_aligned, X4_aligned, X5_aligned, X6_aligned]  # 각각 shape: (T, 3)
X_aligned_trials = np.stack(X_aligned_trials, axis=0)  # shape: (6, T, 3)

# 평균 latent dynamics 계산
X_mean_latent = np.mean(X_aligned_trials, axis=0)  # shape: (T, 3)

# 시각화
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 개별 trajectory
colors = ['crimson', 'crimson', 'crimson','crimson', 'crimson', 'crimson']
for i, traj in enumerate(X_aligned_trials):
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], alpha=0.25, linewidth=2, color=colors[i], label=f"Trial {i+1}")

# 평균 trajectory
ax.plot(X_mean_latent[:, 0], X_mean_latent[:, 1], X_mean_latent[:, 2],
        color='crimson', linewidth=3, linestyle='-', label='Mean trajectory')

ax.set_title("Aligned Latent Dynamics_Mean_stim1~6")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()
plt.tight_layout()

# Save figure
filename = f"stim1~6_mean multi CCA alignment_3D_LD.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()

#########################
# Visualization_stim4

# 세 trial의 aligned latent dynamics (shape: [time_bins, 3])
Y_aligned_trials = [Y1_aligned, Y2_aligned, Y3_aligned, Y4_aligned, Y5_aligned, Y6_aligned]  # 각각 shape: (T, 3)
Y_aligned_trials = np.stack(Y_aligned_trials, axis=0)  # shape: (6, T, 3)

# 평균 latent dynamics 계산
Y_mean_latent = np.mean(Y_aligned_trials, axis=0)  # shape: (T, 3)

# 시각화
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 개별 trajectory
colors = ['royalblue', 'royalblue', 'royalblue', 'royalblue', 'royalblue', 'royalblue']
for i, traj in enumerate(Y_aligned_trials):
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], alpha=0.25, linewidth=2, color=colors[i], label=f"Trial {i+1}")

# 평균 trajectory
ax.plot(Y_mean_latent[:, 0], Y_mean_latent[:, 1], Y_mean_latent[:, 2],
        color='royalblue', linewidth=3, linestyle='-', label='Mean trajectory')

ax.set_title("Aligned Latent Dynamics_Mean_stim7~12")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()
plt.tight_layout()

# Save figure
filename = f"stim7~12_mean multi CCA alignment_3D_LD.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot mean latent trajectory for stim3
ax.plot(X_mean_latent[:, 0], X_mean_latent[:, 1], X_mean_latent[:, 2],
        color='crimson', linewidth=3, linestyle='-', label='Stimulus 3')

# Plot mean latent trajectory for stim4
ax.plot(Y_mean_latent[:, 0], Y_mean_latent[:, 1], Y_mean_latent[:, 2],
        color='royalblue', linewidth=3, linestyle='-', label='Stimulus 4')

# Plot mean latent trajectory for stim5
#ax.plot(Z_mean_latent[:, 0], Z_mean_latent[:, 1], Z_mean_latent[:, 2],
        #color='seagreen', linewidth=3, linestyle='-', label='Stimulus 5')

# Set axis labels and legend
ax.set_title("Mean Latent Dynamics (Aligned via CCA)", fontsize=14)
ax.set_xlabel("LD1", fontsize=12)
ax.set_ylabel("LD2", fontsize=12)
ax.set_zlabel("LD3", fontsize=12)
ax.legend()
ax.grid(True)

plt.tight_layout()

# Save figure
filename = f"Mean_Latent_Dynamics_All_Stimuli_3D_LD.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
print(f"[saved] {save_path}")

plt.show()
plt.close()

In [ ]:
def smooth_trajectory(traj, n_interp=100):
    t = np.linspace(0, 1, traj.shape[0])
    t_new = np.linspace(0, 1, n_interp)
    spl_x = make_interp_spline(t, traj[:, 0], k=3)
    spl_y = make_interp_spline(t, traj[:, 1], k=3)
    spl_z = make_interp_spline(t, traj[:, 2], k=3)
    x_smooth = spl_x(t_new)
    y_smooth = spl_y(t_new)
    z_smooth = spl_z(t_new)
    return np.stack([x_smooth, y_smooth, z_smooth], axis=1)


#########################
# Visualization_stim3

# 세 trial의 aligned latent dynamics (shape: [time_bins, 3])
X_aligned_trials = [X1_aligned, X2_aligned, X3_aligned, X4_aligned, X5_aligned, X6_aligned]  # 각각 shape: (T, 3)
X_aligned_trials_smooth = [smooth_trajectory(traj, n_interp=200) for traj in X_aligned_trials]
X_aligned_trials_smooth = np.stack(X_aligned_trials_smooth, axis=0)  # shape: (6, T, 3)


# 평균 latent dynamics 계산
X_mean_latent = np.mean(X_aligned_trials_smooth, axis=0)

# 시각화
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 개별 trajectory
colors = ['crimson', 'crimson', 'crimson', 'crimson', 'crimson', 'crimson']
for i, traj in enumerate(X_aligned_trials_smooth):
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], alpha=0.25, linewidth=2, color=colors[i], label=f"Trial {i+1}")

# 평균 trajectory
ax.plot(X_mean_latent[:, 0], X_mean_latent[:, 1], X_mean_latent[:, 2],
        color='crimson', linewidth=3, linestyle='-', label='Mean trajectory')

ax.set_title("Aligned Latent Dynamics_Mean_stim3_smooth")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()
plt.tight_layout()

# Save figure
filename = f"stim1~6_mean multi CCA alignment_3D_LD_smooth.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()

#########################
# Visualization_stim4

# 세 trial의 aligned latent dynamics (shape: [time_bins, 3])
Y_aligned_trials = [Y1_aligned, Y2_aligned, Y3_aligned, Y4_aligned, Y5_aligned, Y6_aligned]  # 각각 shape: (T, 3)
Y_aligned_trials_smooth = [smooth_trajectory(traj, n_interp=200) for traj in Y_aligned_trials]
Y_aligned_trials_smooth = np.stack(Y_aligned_trials_smooth, axis=0)  # shape: (6, T, 3)

# 평균 latent dynamics 계산
Y_mean_latent = np.mean(Y_aligned_trials_smooth, axis=0)  


# 시각화
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 개별 trajectory
colors = ['royalblue', 'royalblue', 'royalblue', 'royalblue', 'royalblue', 'royalblue']
for i, traj in enumerate(Y_aligned_trials_smooth):
    ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], alpha=0.25, linewidth=2, color=colors[i], label=f"Trial {i+1}")

# 평균 trajectory
ax.plot(Y_mean_latent[:, 0], Y_mean_latent[:, 1], Y_mean_latent[:, 2],
        color='royalblue', linewidth=3, linestyle='-', label='Mean trajectory')

ax.set_title("Aligned Latent Dynamics_Mean_stim4")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()
plt.tight_layout()

# Save figure
filename = f"stim7~12_mean multi CCA alignment_3D_LD_smooth.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

ax.plot(X_mean_latent[:, 0], X_mean_latent[:, 1], X_mean_latent[:, 2], color='crimson', label='Stimulus 1 (mean)', linewidth=2.5)
ax.plot(Y_mean_latent[:, 0], Y_mean_latent[:, 1], Y_mean_latent[:, 2], color='royalblue', label='Stimulus 2 (mean)', linewidth=2.5)
#ax.plot(Z_mean_latent[:, 0], Z_mean_latent[:, 1], Z_mean_latent[:, 2], color='seagreen', label='Stimulus 3 (mean)', linewidth=2.5)

ax.set_title("Smoothed Mean Latent Dynamics (multi-stimuli)")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()
ax.grid(True)
plt.tight_layout()

# Save figure
filename = f"Mean_Latent_Dynamics_All_Stimuli_3D_LD_sm.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
print(f"[saved] {save_path}")

plt.show()

In [ ]:
X1_smooth = smooth_trajectory(X1_aligned, n_interp=200)
X2_smooth = smooth_trajectory(X2_aligned, n_interp=200)
X3_smooth = smooth_trajectory(X3_aligned, n_interp=200)
X4_smooth = smooth_trajectory(X4_aligned, n_interp=200)
X5_smooth = smooth_trajectory(X5_aligned, n_interp=200)
X6_smooth = smooth_trajectory(X6_aligned, n_interp=200)

Y1_smooth = smooth_trajectory(Y1_aligned, n_interp=200)
Y2_smooth = smooth_trajectory(Y2_aligned, n_interp=200)
Y3_smooth = smooth_trajectory(Y3_aligned, n_interp=200)
Y4_smooth = smooth_trajectory(Y4_aligned, n_interp=200)
Y5_smooth = smooth_trajectory(Y5_aligned, n_interp=200)
Y6_smooth = smooth_trajectory(Y6_aligned, n_interp=200)

# ---------- Visualization_stim3 (1~6) : 스무딩 버전 ----------
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 팔레트(같은 계열색: Blues)
trial_colors_1to6 = ['#9ecae1', '#6baed6', '#4292c6', '#2171b5', '#08519c', '#08306b']

ax.plot(X1_smooth[:, 0], X1_smooth[:, 1], X1_smooth[:, 2], linewidth=2, color='#9ecae1', label=f'{stimulus1}')
for i in range(1, len(X1_smooth)):
    ax.plot([X1_smooth[i-1, 0], X1_smooth[i, 0]],
            [X1_smooth[i-1, 1], X1_smooth[i, 1]],
            [X1_smooth[i-1, 2], X1_smooth[i, 2]],
            color=plt.cm.Blues(i / len(X1_smooth)))

ax.plot(X2_smooth[:, 0], X2_smooth[:, 1], X2_smooth[:, 2], linewidth=2, color='#6baed6', label=f'{stimulus2}')
for i in range(1, len(X2_smooth)):
    ax.plot([X2_smooth[i-1, 0], X2_smooth[i, 0]],
            [X2_smooth[i-1, 1], X2_smooth[i, 1]],
            [X2_smooth[i-1, 2], X2_smooth[i, 2]],
            color=plt.cm.Reds(i / len(X2_smooth)))

ax.plot(X3_smooth[:, 0], X3_smooth[:, 1], X3_smooth[:, 2], linewidth=2, color='#4292c6', label=f'{stimulus3}')
for i in range(1, len(X3_smooth)):
    ax.plot([X3_smooth[i-1, 0], X3_smooth[i, 0]],
            [X3_smooth[i-1, 1], X3_smooth[i, 1]],
            [X3_smooth[i-1, 2], X3_smooth[i, 2]],
            color=plt.cm.Greens(i / len(X3_smooth)))

ax.plot(X4_smooth[:, 0], X4_smooth[:, 1], X4_smooth[:, 2], linewidth=2, color='#2171b5', label=f'{stimulus4}')
for i in range(1, len(X4_smooth)):
    ax.plot([X4_smooth[i-1, 0], X4_smooth[i, 0]],
            [X4_smooth[i-1, 1], X4_smooth[i, 1]],
            [X4_smooth[i-1, 2], X4_smooth[i, 2]],
            color=plt.cm.Oranges(i / len(X4_smooth)))

ax.plot(X5_smooth[:, 0], X5_smooth[:, 1], X5_smooth[:, 2], linewidth=2, color='#08519c', label=f'{stimulus5}')
for i in range(1, len(X5_smooth)):
    ax.plot([X5_smooth[i-1, 0], X5_smooth[i, 0]],
            [X5_smooth[i-1, 1], X5_smooth[i, 1]],
            [X5_smooth[i-1, 2], X5_smooth[i, 2]],
            color=plt.cm.Purples(i / len(X5_smooth)))

ax.plot(X6_smooth[:, 0], X6_smooth[:, 1], X6_smooth[:, 2], linewidth=2, color='#08306b', label=f'{stimulus6}')
for i in range(1, len(X6_smooth)):
    ax.plot([X6_smooth[i-1, 0], X6_smooth[i, 0]],
            [X6_smooth[i-1, 1], X6_smooth[i, 1]],
            [X6_smooth[i-1, 2], X6_smooth[i, 2]],
            color=plt.cm.BuGn(i / len(X6_smooth)))

ax.set_title(f"Multiset CCA-aligned Latent Dynamics_stim3 (smoothed)")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()

filename = f"stim3_multi CCA alignment_3D_LD_smooth.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()

# ---------- Visualization_stim4 (7~12) : 스무딩 버전 ----------
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 팔레트(같은 계열색: Oranges)
trial_colors_7to12 = ['#fdd0a2', '#fdae6b', '#fd8d3c', '#f16913', '#d94801', '#a63603']

ax.plot(Y1_smooth[:, 0], Y1_smooth[:, 1], Y1_smooth[:, 2], linewidth=2, color='#fdd0a2', label=f'{stimulus7}')
for i in range(1, len(Y1_smooth)):
    ax.plot([Y1_smooth[i-1, 0], Y1_smooth[i, 0]],
            [Y1_smooth[i-1, 1], Y1_smooth[i, 1]],
            [Y1_smooth[i-1, 2], Y1_smooth[i, 2]],
            color=plt.cm.Blues(i / len(Y1_smooth)))

ax.plot(Y2_smooth[:, 0], Y2_smooth[:, 1], Y2_smooth[:, 2], linewidth=2, color='#fdae6b', label=f'{stimulus8}')
for i in range(1, len(Y2_smooth)):
    ax.plot([Y2_smooth[i-1, 0], Y2_smooth[i, 0]],
            [Y2_smooth[i-1, 1], Y2_smooth[i, 1]],
            [Y2_smooth[i-1, 2], Y2_smooth[i, 2]],
            color=plt.cm.Reds(i / len(Y2_smooth)))

ax.plot(Y3_smooth[:, 0], Y3_smooth[:, 1], Y3_smooth[:, 2], linewidth=2, color='#fd8d3c', label=f'{stimulus9}')
for i in range(1, len(Y3_smooth)):
    ax.plot([Y3_smooth[i-1, 0], Y3_smooth[i, 0]],
            [Y3_smooth[i-1, 1], Y3_smooth[i, 1]],
            [Y3_smooth[i-1, 2], Y3_smooth[i, 2]],
            color=plt.cm.Greens(i / len(Y3_smooth)))

ax.plot(Y4_smooth[:, 0], Y4_smooth[:, 1], Y4_smooth[:, 2], linewidth=2, color='#f16913', label=f'{stimulus10}')
for i in range(1, len(Y4_smooth)):
    ax.plot([Y4_smooth[i-1, 0], Y4_smooth[i, 0]],
            [Y4_smooth[i-1, 1], Y4_smooth[i, 1]],
            [Y4_smooth[i-1, 2], Y4_smooth[i, 2]],
            color=plt.cm.Blues(i / len(Y4_smooth)))

ax.plot(Y5_smooth[:, 0], Y5_smooth[:, 1], Y5_smooth[:, 2], linewidth=2, color='#d94801', label=f'{stimulus11}')
for i in range(1, len(Y5_smooth)):
    ax.plot([Y5_smooth[i-1, 0], Y5_smooth[i, 0]],
            [Y5_smooth[i-1, 1], Y5_smooth[i, 1]],
            [Y5_smooth[i-1, 2], Y5_smooth[i, 2]],
            color=plt.cm.Reds(i / len(Y5_smooth)))

ax.plot(Y6_smooth[:, 0], Y6_smooth[:, 1], Y6_smooth[:, 2], linewidth=2, color='#a63603', label=f'{stimulus12}')
for i in range(1, len(Y6_smooth)):
    ax.plot([Y6_smooth[i-1, 0], Y6_smooth[i, 0]],
            [Y6_smooth[i-1, 1], Y6_smooth[i, 1]],
            [Y6_smooth[i-1, 2], Y6_smooth[i, 2]],
            color=plt.cm.Greens(i / len(Y6_smooth)))

ax.set_title(f"Multiset CCA-aligned Latent Dynamics_stim4 (smoothed)")
ax.set_xlabel("LD1")
ax.set_ylabel("LD2")
ax.set_zlabel("LD3")
ax.legend()

filename = f"stim4_multi CCA alignment_3D_LD_smooth.svg"
save_path = os.path.join(output_dir, filename)
plt.savefig(save_path, format='svg', bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
def pairwise_cca_mean_r2(data_list, name_list, n_components=3):
    """
    data_list: [np.ndarray(T,3), ...] 길이 6
    name_list: [stimulus_str, ...]    길이 6
    return: 결과 DataFrame
    """
    results = []

    # 모든 쌍 (C(6,2)=15)
    idx_pairs = list(combinations(range(len(data_list)), 2))
    for (i, j) in idx_pairs:
        A = data_list[i]
        B = data_list[j]

        # 길이 맞추기 (시간축 최소 길이)
        T = min(A.shape[0], B.shape[0])
        A = A[:T, :3]
        B = B[:T, :3]

        # 표준화(각 축 z-score) -> CCA(scale=False)
        scalerA = StandardScaler()
        scalerB = StandardScaler()
        A_z = scalerA.fit_transform(A)
        B_z = scalerB.fit_transform(B)

        cca = CCA(n_components=n_components, scale=False, max_iter=5000)
        cca.fit(A_z, B_z)
        U, V = cca.transform(A_z, B_z)  # (T, n_components)

        # 정준상관계수 계산 (축별)
        rhos = []
        rhos2 = []
        for k in range(n_components):
            r = np.corrcoef(U[:, k], V[:, k])[0, 1]
            # 수치오차 방지용 clip
            r = float(np.clip(r, -1.0, 1.0))
            rhos.append(r)
            rhos2.append(r * r)

        mean_r2 = float(np.mean(rhos2))

        results.append({
            'Pair': f'{name_list[i]} - {name_list[j]}',
            'Stim1': name_list[i],
            'Stim2': name_list[j],
            'rho1': rhos[0],
            'rho2': rhos[1] if n_components > 1 else np.nan,
            'rho3': rhos[2] if n_components > 2 else np.nan,
            'rho1^2': rhos2[0],
            'rho2^2': rhos2[1] if n_components > 1 else np.nan,
            'rho3^2': rhos2[2] if n_components > 2 else np.nan,
            'mean_rho^2_(1-3)': mean_r2
        })

    df = pd.DataFrame(results)
    # 보기 좋게 평균 rho^2 기준 내림차순 정렬
    df = df.sort_values('mean_rho^2_(1-3)', ascending=False).reset_index(drop=True)
    return df

# ---------------------------------
# 그룹 A (stimulus1~6; X1~X6_aligned)
# ---------------------------------
groupA_data = [X1_aligned, X2_aligned, X3_aligned, X4_aligned, X5_aligned, X6_aligned]
groupA_names = [stimulus1, stimulus2, stimulus3, stimulus4, stimulus5, stimulus6]

df_groupA = pairwise_cca_mean_r2(groupA_data, groupA_names, n_components=3)

excel_path = os.path.join(output_dir, 'groupA_pairwise_CCA_meanrho2.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_groupA.to_excel(writer, index=False, sheet_name='pairwise_mean_r2')

print(f'[그룹 A] pairwise mean rho^2 결과를 {excel_path} 에 저장했습니다.')

# ---------------------------------
# 그룹 B (stimulus7~12; Y1~Y6_aligned)
# ---------------------------------
groupB_data = [Y1_aligned, Y2_aligned, Y3_aligned, Y4_aligned, Y5_aligned, Y6_aligned]
groupB_names = [stimulus7, stimulus8, stimulus9, stimulus10, stimulus11, stimulus12]

df_groupB = pairwise_cca_mean_r2(groupB_data, groupB_names, n_components=3)

excel_path = os.path.join(output_dir, 'groupB_pairwise_CCA_meanrho2.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_groupB.to_excel(writer, index=False, sheet_name='pairwise_mean_r2')

print(f'[그룹 B] pairwise mean rho^2 결과를 {excel_path} 에 저장했습니다.')